# 01 · Topic classification | Riwaq AI

**Purpose.** Assign up to six topics to an English educational post. A post can have several topics or remain unclassified when confidence is insufficient.

**Reading guide.** This is a curated record of the work, with selected, lightly shortened code excerpts and recorded results. It is not a training pipeline to rerun cell by cell: the full run depended on Kaggle datasets, checkpoints, and GPU resources. New code cells have intentionally empty execution outputs. All numerical results below come from the executed source notebook or the model report in the repository.

**Evidence:** `content-classification.ipynb` (original training, cells 3–36); `notebookfa80b851c2.ipynb` (later experiments, cells 4–25); `../app/models/topic_model/evaluation_report.json` (deployed model).

## 1. Define the task before choosing the model

The post may discuss *robotics and electronics* together, so I formulated topic prediction as **multilabel classification**. I defined 12 stable product topics and used independent sigmoid scores for each. The output policy caps selected topics at six and can return `Unclassified`; a forced top-one choice would hide uncertainty.

In [ ]:
LABELS = [
    "PROGRAMMING_WEB", "AI_DATA", "ELECTRONICS_EMBEDDED", "ROBOTICS",
    "CYBERSECURITY", "DESIGN", "MATHEMATICS", "NATURAL_SCIENCES",
    "HEALTH_MEDICINE", "BUSINESS_ECONOMICS", "LANG_COMMUNICATION",
    "HUMANITIES_SOCIAL",
]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
MAX_RETURNED_TOPICS = 6
# Original: content-classification.ipynb, cell 3.

## 2. Collect, map, and audit the data

The original run mapped **arXiv categories** and **Yahoo Answers Topics** into the product taxonomy, and included `tanaos/synthetic-topic-classification-dataset-v1` with a synthetic-data flag. The source label, source ID, mapping method, and confidence were retained. Yahoo's dataset card did not state a license in the run's report, so this source was marked for company review before production use.

I normalized text, removed very short or long examples, deduplicated by a stable text hash, and preferred higher-confidence real labels when duplicate texts conflicted. This matters because duplicates across training and evaluation can inflate the score. The split was multilabel-stratified on real records; synthetic examples joined **training only**.

In [ ]:
# Selected split logic from the original training cell (cell 15).
# real_df and synthetic_df are the cleaned frames; y holds real multilabel vectors.
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import numpy as np
import pandas as pd

SEED = 42
y = np.stack(real_df["label_vector"].to_numpy())
split_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(split_1.split(np.zeros(len(y)), y))
split_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED + 1)
val_local, test_local = next(split_2.split(np.zeros(len(temp_idx)), y[temp_idx]))

train_df = pd.concat([real_df.iloc[train_idx], synthetic_df], ignore_index=True)
val_df = real_df.iloc[temp_idx[val_local]].copy()
test_df = real_df.iloc[temp_idx[test_local]].copy()
assert set(train_df.text_hash).isdisjoint(val_df.text_hash)
assert set(train_df.text_hash).isdisjoint(test_df.text_hash)
assert not val_df.is_synthetic.any() and not test_df.is_synthetic.any()

**Recorded split:** 273,295 training · 33,202 validation · 33,203 test rows. These counts describe the merged dataset; transformer training used a balanced **60,000-row subset** of training to fit GPU resources. Source: original notebook, cells 15–17.

## 3. Train and select on validation

I evaluated a transformer approach, with **DeBERTa v3 base** and **ModernBERT base** scheduled in the source. The saved leaderboard contains one successfully evaluated candidate, DeBERTa; it does **not** establish a measured comparison against ModernBERT. Positive-class weights, capped at 8, reduced the effect of topic imbalance. The training configuration used three epochs, learning rate `2e-5`, weight decay `0.01`, cosine scheduling and best checkpoint selection by validation Micro-F1. This is why selection follows validation rather than the test set.

In [ ]:
# Core loss from content-classification.ipynb, cell 22 (surrounding Trainer setup omitted).
import torch.nn.functional as F
from transformers import Trainer

class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        loss = F.binary_cross_entropy_with_logits(
            outputs.logits, labels, pos_weight=self.pos_weight.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss

## 4. Calibrate probabilities and inspect failure cases

A single `0.5` cutoff is not equally suitable for all 12 topics. On validation data I searched a separate threshold for each label by F1; after thresholding, I kept at most the six highest-scoring labels. I measured Micro-F1 for total prediction quality, Macro-F1 to give rarer topics weight, and exact match for posts where the entire topic set is correct. I then inspected per-topic F1 and individual mistakes rather than relying on one aggregate number.

In [ ]:
# Shortened from content-classification.ipynb, cell 20.
import numpy as np
from sklearn.metrics import f1_score

def choose_thresholds(y_validation, probabilities):
    thresholds = np.full(len(LABELS), 0.50)
    for j in range(len(LABELS)):
        if y_validation[:, j].sum() == 0:
            continue
        thresholds[j] = max(
            np.arange(0.10, 0.91, 0.02),
            key=lambda t: f1_score(
                y_validation[:, j], probabilities[:, j] >= t, zero_division=0
            ),
        )
    return thresholds

def select_topics(probabilities, thresholds):
    selected = np.flatnonzero(probabilities >= thresholds)
    return selected[np.argsort(probabilities[selected])[::-1][:MAX_RETURNED_TOPICS]]

### Recorded outcome for the model currently in this repository

| Measure | Validation | Held-out test |
|---|---:|---:|
| Micro-F1 | 0.8714 | 0.8734 |
| Macro-F1 | 0.7869 | 0.7902 |
| Micro-precision | 0.8572 | 0.8587 |
| Micro-recall | 0.8860 | 0.8886 |
| Exact match | 0.8276 | 0.8286 |

The original leaderboard selected `microsoft__deberta_v3_base`. The ensemble option required an improvement of at least **0.015 validation Micro-F1**, and the saved selection remained the single model. On held-out test, `BUSINESS_ECONOMICS` (F1 ≈ 0.579) and `ELECTRONICS_EMBEDDED` (F1 ≈ 0.599) were weaker than `NATURAL_SCIENCES` (F1 ≈ 0.975); aggregate success did not mean every topic was solved.

**Source:** original notebook cells 26–32 and `../app/models/topic_model/evaluation_report.json`.

## 5. What I learned from realistic post tests

The first exported model confidently labeled a *phishing and malware* post as `PROGRAMMING_WEB` (0.9763), with `CYBERSECURITY` at 0.0067. It returned `Unclassified` for an ESP32 robot example despite `ROBOTICS` scoring 0.5191. These are recorded inference outputs, **not** extra held-out metrics (original notebook, cell 36). They motivated domain-specific follow-up experiments.

The second source notebook explored adapting the model with 12,642 synthetic topic rows and manually constructed hard cases. It recorded 8,592 training and 2,025 validation examples for one adaptation stage. On **that stage's validation data**, Micro-F1 rose from 0.5652 (old model) to 0.9376 (adapted model), and one later threshold calibration reported 0.9677. These scores are from a **different evaluation set** than the 33,203-row held-out test above and should never be compared as if they were the same benchmark. Subsequent V5/V6 corrections and an eight-case long-post check were exploratory; the V6 long-post Top-1 result was **0.875 on eight hand-written posts**. No evidence in the repository shows that V6 replaced the deployed DeBERTa bundle.

**Decision:** keep the repository's verifiable held-out result as the deployed baseline; treat the later adaptations as documented experiments requiring a common independent test before promotion.

## 6. Product handoff and limits

The deployed bundle provides `label_map.json`, per-label `thresholds.json`, inference code and an evaluation report. The service uses these to return primary and secondary topics, confidence scores, and an unclassified outcome. This notebook documents **English text** topic classification. It does not establish an evaluated image classifier or prove that the exploratory V6 weights are deployed. Dataset license review remains a requirement for production use.

**Trace:** source notebooks `content-classification.ipynb` and `notebookfa80b851c2.ipynb`; application bundle `../app/models/topic_model/`.